# Практична робота №13
## Точкові оцінки та перевірка гіпотез: z-статистика, p-value, довірчий інтервал

**Виконав:** Панасюк Дмитро  
**Номер за списком:** 15  
**Варіант:** №5 — **Дніпро**  
**Параметр генерації липневих температур:** 23 °C  
**Стара кліматична норма:** $\mu_0 = 21.6$ °C  
**Рівень значущості:** $\alpha = 0.05$

**Мета:** перевірити на синтетичній вибірці, чи є статистичні підстави сумніватися в старій нормі липневої температури, та зіставити z-тест із 95% довірчим інтервалом. Значення вибірки навчальні й не є реальними кліматичними вимірюваннями.

### Аналітичний план — Завдання 2 (формулювання до аналізу даних)

- **Нульова гіпотеза $H_0$:** $\mu = 21.6$ °C — стара середня липнева температура Дніпра залишається правильною.
- **Альтернативна гіпотеза $H_1$:** $\mu \ne 21.6$ °C — середня липнева температура відрізняється від старої норми.
- **Тип тесту:** **двобічний**. У питанні «стара норма більше не відповідає дійсності» важливі як потепління, так і похолодання; тому напрямок не обираємо за результатом вибірки.
- **Правило рішення (зафіксоване наперед):** якщо $p < 0.05$, відхиляємо $H_0$, інакше не маємо достатніх підстав її відхилити.

Далі створюємо дані й виконуємо всі обчислення згідно із зафіксованим планом.

## Завдання 1. Вибірка й точкові оцінки

Генеруємо 30 синтетичних денних температур для липня (центр 23 °C, стандартне відхилення генератора 2.5 °C). Для відтворюваності використано `np.random.seed(5)` — номер **варіанта №5**. Вибіркове стандартне відхилення рахуємо з `ddof=1`.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

# Варіант №5 — Дніпро
variant = 5
city = "Дніпро"
july_temp = 23.0
mu0 = 21.6
alpha = 0.05

np.random.seed(variant)
daily_temps = np.random.normal(loc=july_temp, scale=2.5, size=30)

n = len(daily_temps)
x_bar = daily_temps.mean()
s = daily_temps.std(ddof=1)

print("Вибірка 30 денних температур, °C:")
print(np.array2string(daily_temps, precision=2, separator=", "))
print(f"\nn = {n}")
print(f"Вибіркове середнє x̄ = {x_bar:.4f} °C")
print(f"Вибіркове стандартне відхилення s = {s:.4f} °C")
print(f"Різниця x̄ − μ₀ = {x_bar - mu0:.4f} °C")

Вибірка 30 денних температур, °C:
[24.1 , 22.17, 29.08, 22.37, 23.27, 26.96, 20.73, 21.52, 23.47, 22.18,
 20.02, 22.49, 22.1 , 24.51, 18.84, 21.25, 25.88, 27.64, 19.22, 24.61,
 20.55, 20.86, 20.82, 21.94, 25.49, 24.78, 23.15, 22.09, 23.01, 22.74]

n = 30
Вибіркове середнє x̄ = 22.9277 °C
Вибіркове стандартне відхилення s = 2.4151 °C
Різниця x̄ − μ₀ = 1.3277 °C


**Пояснення:** $\bar{x}$ — точкова оцінка поточного середнього, а $s$ — вибіркова оцінка стандартного відхилення. Вони можуть відрізнятися від параметрів генератора, оскільки $n=30$ — скінченна випадкова вибірка.

## Завдання 2. Формулювання гіпотез

Як зафіксовано **перед генерацією та переглядом числових результатів**, використовуємо $H_0: \mu = 21.6$ °C і $H_1: \mu \ne 21.6$ °C. Тест двобічний, бо розглядаємо будь-яке відхилення від старої норми. Однобічний тест був би обґрунтований лише за завчасно сформульованого питання саме про потепління або похолодання.

## Завдання 3. Стандартна похибка, z-статистика й p-value

Згідно з умовою практики застосовуємо

$$SE = \frac{s}{\sqrt{n}}, \qquad z = \frac{\bar{x} - \mu_0}{SE}, \qquad p = 2\bigl(1-\Phi(|z|)\bigr).$$

У Python `stats.norm.sf(abs(z))` чисельно обчислює $1-\Phi(|z|)$; множник 2 враховує **обидва хвости** розподілу.

In [2]:
se = s / np.sqrt(n)
z_stat = (x_bar - mu0) / se
p_value = 2 * stats.norm.sf(abs(z_stat))

print(f"Стандартна похибка SE = {se:.6f} °C")
print(f"z-статистика = {z_stat:.6f}")
print(f"Двобічне p-value = {p_value:.8f}")

# Контроль збігу з формулою через CDF
p_via_cdf = 2 * (1 - stats.norm.cdf(abs(z_stat)))
assert np.isclose(p_value, p_via_cdf)
assert n == 30

Стандартна похибка SE = 0.440938 °C
z-статистика = 3.011121
Двобічне p-value = 0.00260285


**Інтерпретація p-value.** Якщо стара норма 21.6 °C насправді правильна і припущення z-моделі виконуються, то отримати середнє з таким самим або більшим за модулем відхилення (відносно стандартної похибки) було б малоймовірно. **p-value не є ймовірністю правильності або неправильності $H_0$.**

## Завдання 4. Рішення за рівня значущості α = 0.05

Порівнюємо p-value з наперед заданою межею $\alpha=0.05$ і формулюємо статистичний висновок.

In [3]:
reject_h0 = p_value < alpha
print(f"p-value = {p_value:.6f}; α = {alpha:.2f}")
print("Рішення:", "відхиляємо H₀" if reject_h0 else "не відхиляємо H₀")

if reject_h0:
    print(
        f"Висновок: для Дніпра вибіркове середнє {x_bar:.2f} °C "
        f"статистично значущо відрізняється від старої норми {mu0:.1f} °C "
        f"(двобічний z-тест, z = {z_stat:.3f}, p = {p_value:.4f}, α = {alpha:.2f}). "
        "Спостережена різниця спрямована в бік вищої температури."
    )
else:
    print(
        f"Висновок: для Дніпра немає достатніх статистичних підстав "
        f"вважати, що середня липнева температура відрізняється від {mu0:.1f} °C "
        f"(двобічний z-тест, p = {p_value:.4f}, α = {alpha:.2f})."
    )

p-value = 0.002603; α = 0.05
Рішення: відхиляємо H₀
Висновок: для Дніпра вибіркове середнє 22.93 °C статистично значущо відрізняється від старої норми 21.6 °C (двобічний z-тест, z = 3.011, p = 0.0026, α = 0.05). Спостережена різниця спрямована в бік вищої температури.


## Завдання 5. Довірчий інтервал і помилки I/II роду

### 5.1. Порівняння 95% z-інтервалу з тестом

За умовою практики використовуємо **той самий нормальний підхід**, що й у z-тесті:

$$CI_{95\%} = \bar{x} \pm 1.96\cdot SE.$$

Для узгодження з двобічним тестом критичне значення беремо як `stats.norm.ppf(0.975)` (приблизно 1.96). Якщо $\mu_0$ не входить у 95% інтервал, двобічний z-тест має відхилити $H_0$ при $\alpha=0.05$.

In [4]:
z_crit = stats.norm.ppf(1 - alpha / 2)
ci_lower = x_bar - z_crit * se
ci_upper = x_bar + z_crit * se
mu0_in_ci = ci_lower <= mu0 <= ci_upper

print(f"Критичне z = {z_crit:.5f}")
print(f"95% довірчий інтервал: [{ci_lower:.4f}; {ci_upper:.4f}] °C")
print(f"Чи потрапляє μ₀ = {mu0:.1f} °C в інтервал? {'Так' if mu0_in_ci else 'Ні'}")
print(f"Чи узгоджуються висновки CI та z-тесту? {'Так' if (reject_h0 == (not mu0_in_ci)) else 'Ні'}")
assert reject_h0 == (not mu0_in_ci)

Критичне z = 1.95996
95% довірчий інтервал: [22.0635; 23.7919] °C
Чи потрапляє μ₀ = 21.6 °C в інтервал? Ні
Чи узгоджуються висновки CI та z-тесту? Так


**Результат для Дніпра:** вибіркове середнє приблизно **22.93 °C**; 95% z-інтервал приблизно **[22.06; 23.79] °C**. Стара норма **21.6 °C** лежить поза інтервалом, тому інтервальна перевірка узгоджується з відхиленням $H_0$ у двобічному тесті.

**Методологічне зауваження:** у цій практиці **за прямою вимогою** застосовано z-статистику і z-інтервал з оцінкою $s$. Коли генеральне $\sigma$ невідоме (як тут), класичний висновок для середнього невеликої нормальної вибірки зазвичай спирається на **t-розподіл** з $n-1=29$ ступенями свободи (Практика 12). Це не змінює висновку для даного набору, але означає, що наведена z-процедура — саме навчальне нормальне наближення із завдання.

### 5.2. Помилки I та II роду

**Помилка I роду (хибне відхилення $H_0$).** Статистично оголосили б, що середня липнева температура Дніпра **відрізняється** від 21.6 °C, хоча справжня середня й досі дорівнює 21.6 °C. Це хибна тривога і ризик безпідставно оновити офіційний довідник.

**Помилка II роду (хибне невідхилення $H_0$).** Не знайшли б достатніх доказів відмінності від 21.6 °C, хоча справжня середня температура Дніпра **вже змінилася**. Це ризик залишити в довіднику застарілу норму та пропустити зміну клімату.

**Яка дорожча?** Для відповідальних за публікацію офіційної кліматичної норми особливо небезпечно **хибно змінити довідник** на підставі випадкових коливань (помилка I роду); водночас пропустити реальну зміну (помилка II роду) теж важливо. Остаточний пріоритет залежить від призначення довідника, тому для реального рішення потрібні незалежні спостереження за багато років.

## Контрольні питання — письмові відповіді

**1. Що означає отримане p-value і чому це не ймовірність правильності старої норми?**  
Для Дніпра $p \approx 0.0026$: **за умови**, що справжнє середнє дорівнює 21.6 °C, спостерігати таке або ще сильніше за модулем стандартизоване відхилення середнього від норми було б досить рідко. p-value оцінює узгодженість даних із $H_0$ за моделлю, але не обчислює $P(H_0\mid \text{дані})$.

**2. Чому потрібно обрати однобічний або двобічний тест до обчислень?**  
Вибір напрямку визначає, яка саме частина розподілу потрапляє в p-value. Якщо обирати тест після перегляду середнього, можна штучно зменшити p-value та завищити ризик помилки I роду. Тут до аналізу зафіксовано **двобічний** тест, бо будь-яке відхилення від 21.6 °C є предметом перевірки.

**3. Що означають помилки I і II роду для нашого набору?**  
Помилка I роду — заявити, що стара норма 21.6 °C для Дніпра більше не правильна, коли вона насправді правильна. Помилка II роду — не відхилити стару норму, коли реальна середня температура вже від неї відрізняється. У першому випадку це **хибна тривога**, у другому — **пропущена зміна**.

**4. Чи узгодилися тест і довірчий інтервал, і чому?**  
**Так.** Двобічний z-тест дав $p \approx 0.0026 < 0.05$, тож $H_0$ відхилено; водночас $\mu_0 = 21.6$ °C не входить до 95% z-інтервалу **[22.06; 23.79] °C**. Для **двобічного** тесту і відповідного **двобічного** інтервалу, побудованих за однією моделлю на тому самому рівні $\alpha=0.05$, ці рішення математично еквівалентні (для однобічного тесту потрібен відповідний однобічний інтервал).

## Загальний висновок

У практичній роботі створено відтворювану вибірку 30 синтетичних денних температур Дніпра, обчислено точкові оцінки середнього й стандартного відхилення, стандартну похибку, двобічну z-статистику та p-value. Для старої норми $\mu_0=21.6$ °C отримано статистично значущу відмінність на рівні $\alpha=0.05$; 95% довірчий інтервал підтверджує той самий висновок. Роз'яснено зміст p-value і помилок I/II роду.

**Обмеження:** це симуляція за заданим генератором `N(23, 2.5²)`, а не емпіричне свідчення реального потепління Дніпра.